# training-step-cycle — ex3: diagnose a step-before-backward ordering bug

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `training-step-cycle`. Running the final beacon cell reports progress against the `PyTorch: Training step cycle` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Training step cycle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`training-step-cycle`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "training-step-cycle"
DD_SUBTOPIC = "PyTorch: Training step cycle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Training-step cycle — quick refresher

```
logits = model(x)             # 1. forward
loss   = loss_fn(logits, y)   # 2. loss
loss.backward()               # 3. backward → grads into .grad
optimizer.step()              # 4. apply update from .grad
optimizer.zero_grad()         # 5. clear .grad for next batch
```

**Two ordering invariants.** `backward` must come *before* `step` — otherwise `.grad` is `None` (or stale from a previous batch) and `step` either crashes or applies the wrong update. `zero_grad` must come *after* `step` (or before next forward) so gradients don't accumulate across batches. Swapping `backward` and `step` is the silent-failure variant: no error is raised, but training silently uses the previous step's grad.

### Exercise 3 — diagnose a step-before-backward ordering bug

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze a training loop whose `optimizer.step()` is called BEFORE `loss.backward()` and fix the ordering so the loss curve becomes monotonically decreasing on a 1-parameter regression.
> Keywords: debug, ordering-bug, stale-grad, silent-failure
> ```

**KCs targeted:** `training-step-five-call-order`, `training-step-debug-via-loss-trajectory`

Below is `train_swapped` — a training loop where someone wrote `optimizer.step()` BEFORE `loss.backward()`. No Python exception is raised: on the first iteration `w.grad` is `None`, so `optimizer.step()` is a no-op; on every later iteration `step` consumes the previous iteration's gradient — one step BEHIND.

Implement `ex3_train_fixed(w_init, x, y, lr, n_steps)` — the corrected version. Use the canonical 5-call order:
  `forward → loss → backward → step → zero_grad`

Return `(w_final, fixed_losses, swapped_losses)`:
- `w_final`: a detached 1-element tensor of the trained weight.
- `fixed_losses`: list of `n_steps` floats from YOUR loop.
- `swapped_losses`: list of `n_steps` floats from `train_swapped` on the SAME inputs (call it yourself).

Snapshot every loss BEFORE `backward()` so the trajectory is the pre-step value at iteration `i`. The test verifies your loop converges and the swapped loop lags behind.

In [ ]:
def ex3_train_fixed(w_init, x, y, lr, n_steps):
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    fixed_losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        fixed_losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    _, swapped_losses = train_swapped(w_init, x, y, lr, n_steps)
    return w.detach().clone(), fixed_losses, swapped_losses


<details><summary>Solution</summary>

```python
def ex3_train_fixed(w_init, x, y, lr, n_steps):
    w = t.tensor([w_init], requires_grad=True)
    optimizer = t.optim.SGD([w], lr=lr)
    fixed_losses = []
    for _ in range(n_steps):
        pred = w * x
        loss = ((pred - y) ** 2).mean()
        fixed_losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    _, swapped_losses = train_swapped(w_init, x, y, lr, n_steps)
    return w.detach().clone(), fixed_losses, swapped_losses
```

**Why the swap is a SILENT bug.** On iteration 0, `w.grad is None` — `optimizer.step()` short-circuits with no update. On iteration 1, `step()` consumes iteration-0's gradient (computed AFTER step on iteration 0). The whole run is one step behind.

**No exception is raised** because PyTorch tolerates `grad=None` in `step()` (it just skips that param). Only the loss trajectory exposes the lag — which is why ARENA repeatedly emphasizes 'log your loss every iteration and look at the curve, not just the final value.'

**The cure is mechanical.** Memorize the order `forward → loss → backward → step → zero_grad` as one atomic block. Every legit PyTorch training loop in the wild has this skeleton.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()